# DermaLens eye-bag training (Colab)

Before running:
1. Locally: `python scripts/make_colab_bundle.py --version v1` and upload the zip to your Google Drive root.
2. Runtime -> Change runtime type -> **T4 GPU**.
3. Run cells top to bottom. If the session disconnects mid-training, just re-run — the train cell auto-resumes from `last.pt`.

Expected runtimes on a T4 with ~1.5k crops: binary ~10-20 min, ordinal ~20-35 min.

In [ ]:
BUNDLE_VERSION = 'v1'   # bump after re-annotation / re-split
# Stages: seed_pretrain (throwaway pre-annotator; bundle with --splits data/seed_splits)
#         -> baseline_binary -> ordinal_severity -> multitask
STAGE = 'baseline_binary'

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unpack the bundle and restore any previous experiments from Drive
import os, pathlib, shutil

WORK = '/content/dermalens'
DRIVE_DIR = '/content/drive/MyDrive/dermalens'
bundle = f'/content/drive/MyDrive/dermalens_bundle_{BUNDLE_VERSION}.zip'
assert os.path.exists(bundle), f'Upload {bundle} to your Drive root first'

os.makedirs(WORK, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
!cd {WORK} && unzip -qo {bundle}

# Restore experiments/ from Drive so --resume works across sessions
drive_exp = f'{DRIVE_DIR}/experiments'
if os.path.exists(drive_exp):
    shutil.copytree(drive_exp, f'{WORK}/experiments', dirs_exist_ok=True)
    print('Restored experiments/ from Drive')
%cd {WORK}

In [ ]:
# Dependencies: torch/torchvision/pandas/sklearn are Colab-preinstalled.
# The training import chain needs only these two extras (no mediapipe/cv2).
!pip -q install albumentations pyyaml
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# Sanity check: 2-epoch smoke run on the bundled synthetic data (~1 min)
!python scripts/train.py --config configs/smoke.yaml --smoke

In [ ]:
# Train the selected stage, auto-resuming if a previous session left a last.pt.
# Stage order and warm-starts:
#   0. seed_pretrain     (throwaway pre-annotator, no warm-start)
#   1. baseline_binary   (Milestone 1, monitor val_auroc)
#   2. ordinal_severity  --warm-start experiments/baseline_binary/best.pt
#   3. multitask         --warm-start experiments/ordinal_severity/best.pt
import os

WARM = {
    'baseline_binary': '',
    'ordinal_severity': 'experiments/baseline_binary/best.pt',
    'multitask': 'experiments/ordinal_severity/best.pt',
}.get(STAGE, '')
last = f'experiments/{STAGE}/last.pt'

cmd = f'python scripts/train.py --config configs/{STAGE}.yaml'
if os.path.exists(last):
    cmd += f' --resume {last}'
    print('Resuming from', last)
elif WARM and os.path.exists(WARM):
    cmd += f' --warm-start {WARM}'
    print('Warm-starting from', WARM)
print(cmd)
!{cmd}

In [ ]:
# Back up experiments/ to Drive (run after every training cell!)
!mkdir -p {DRIVE_DIR}/experiments && cp -r experiments/* {DRIVE_DIR}/experiments/
!ls -lh {DRIVE_DIR}/experiments/*/

In [ ]:
# Evaluate on the validation split (iterate here; test_internal only at gates).
# seed_pretrain uses data/seed_splits; all real stages use data/splits.
VAL_CSV = 'data/seed_splits/val.csv' if STAGE == 'seed_pretrain' else 'data/splits/val.csv'
!python scripts/evaluate.py --checkpoint experiments/{STAGE}/best.pt --csv {VAL_CSV}

In [ ]:
# Zip the run for local download (error analysis happens on the laptop)
!zip -qr /content/{STAGE}_run.zip experiments/{STAGE}
from google.colab import files
files.download(f'/content/{STAGE}_run.zip')